In [ ]:
import subprocess, sys, os, glob, shutil, time, json, importlib.util
def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True); print(r.stdout[-1500:], r.stderr[-800:], flush=True); return r
W = "/kaggle/working/w"; shutil.rmtree(W, ignore_errors=True); os.makedirs(W)
src = [p for p in glob.glob("/kaggle/input/**/giai-thich-8-11", recursive=True) if os.path.isdir(p)] or [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/harness/h4_spice_tuple_8_11.py", recursive=True)]
print(src, flush=True); S = src[0]
for z in glob.glob(S + "/*.zip"): sh(f"unzip -o -q {z} -d {W}/{os.path.basename(z)[:-4]}")
for z in glob.glob(S + "/*.tar"):
    d = f"{W}/{os.path.basename(z)[:-4]}"; os.makedirs(d, exist_ok=True); sh(f"tar -xf {z} -C {d}")
    if os.path.isdir(f"{d}/{os.path.basename(d)}"): sh(f"cp -r {d}/{os.path.basename(d)}/. {d}/ && rm -rf {d}/{os.path.basename(d)}")
sh(f"ls -la {S}")
for d in ("harness", "runs"):
    if os.path.isdir(f"{S}/{d}"): shutil.copytree(f"{S}/{d}", f"{W}/{d}", dirs_exist_ok=True)
sh(f"find {W} -type f | head -20; find {W} -type f | wc -l")
assert os.path.exists(f"{W}/harness/h4_spice_tuple_8_11.py"), "DUNG: thieu ma"
sh("apt-get -qq update && apt-get -qq install -y openjdk-8-jre-headless >/dev/null; update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java; java -version")
sh(f"{sys.executable} -m pip install -q pycocoevalcap 2>&1 | tail -2")
SPL = list(importlib.util.find_spec("pycocoevalcap").submodule_search_locations)[0] + "/spice/lib"
for j, sz in {"stanford-corenlp-3.6.0.jar": 7270741, "stanford-corenlp-3.6.0-models.jar": 378116526}.items():
    s = [p for p in glob.glob(f"/kaggle/input/**/{j}", recursive=True)]; assert s, f"DUNG: thieu {j}"
    shutil.copyfile(s[0], f"{SPL}/{j}"); assert os.path.getsize(f"{SPL}/{j}") == sz
print("jar SPICE du", flush=True)
for f in ("giai_thich_8_11.py", "giai_thich_7_11.py", "h4_spice_tuple_7_11.py", "luat_d3.py", "score_run.py", "dg1_cache/test_ac/descriptors.jsonl", "dg1_cache/test_ac/test.jsonl"):
    assert os.path.exists(f"{W}/harness/{f}"), f"DUNG: thieu {f}"
print("so tep raw:", len(glob.glob(f"{W}/runs/**/*_raw.jsonl", recursive=True)), flush=True)
assert len(glob.glob(f"{W}/runs/**/*_raw.jsonl", recursive=True)) == 12


In [ ]:
KQ = {}
def chay(ten, lenh):
    t0 = time.time(); log = f"{W}/{ten}.log"
    p = subprocess.Popen(f"cd {W}/harness && {sys.executable} -u {lenh} > {log} 2>&1", shell=True)
    while p.poll() is None:
        time.sleep(60); print(time.strftime("%H:%M"), ten, open(log, errors="ignore").read()[-200:].replace("\n", " | "), flush=True)
    KQ[ten] = p.returncode; print("=====", ten, "ma thoat", p.returncode, f"{time.time()-t0:.0f} s", flush=True); print(open(log, errors="ignore").read()[-6000:], flush=True)
O = "../runs/luoi/8_11/giai_thich"
chay("giai_thich_8_11", f"giai_thich_8_11.py --out {O}/giai_thich_8_11.json")
chay("h4_thu", f"h4_spice_tuple_8_11.py --n 100 --out {O}/thu/h4.json")
if KQ["h4_thu"] == 0: chay("h4_spice_tuple_8_11", f"h4_spice_tuple_8_11.py --out {O}/h4_spice_tuple_8_11.json")
print(KQ, flush=True)


In [ ]:
R = "/kaggle/working/ket_qua_8_11"; os.makedirs(R, exist_ok=True)
for f in glob.glob(f"{W}/runs/luoi/8_11/giai_thich/**/*.json", recursive=True) + glob.glob(f"{W}/*.log"):
    shutil.copyfile(f, f"{R}/" + ("thu_" if "/thu/" in f else "") + os.path.basename(f))
shutil.make_archive("/kaggle/working/ket_qua_8_11", "zip", R)
shutil.rmtree(W); print(os.listdir(R), KQ, flush=True)
assert all(v == 0 for v in KQ.values()), KQ
